# FIMO Motif-Centered ISM Alignment

Workflow:
1. Extract sequences from CUT&Tag peaks → FASTA (FIMO input)
2. Run FIMO externally
3. Parse FIMO output → motif-centered BED
4. Feed motif-centered BED into `ism.ipynb`

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.infer._utils import parse_bed
from src.data.data_utils import GenomeSequence

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CUTTAG_BED   = 'data/cpp8_peaks.bed'          # CUT&Tag narrowPeak or BED file
FASTA        = '/path/to/genome.fa'

# Sequence window extracted around each peak center (should match model input)
EXTRACT_LEN  = 2000   # bp to extract per peak for FIMO scanning

# FIMO output (run FIMO externally, then set this path)
FIMO_TSV     = 'outputs/fimo/fimo.tsv'

# Output: motif-centered BED (feed this into ism.ipynb as BED)
OUT_BED      = 'data/cpp8_motif_centers.bed'

# FASTA output for FIMO input
OUT_FASTA    = 'outputs/fimo/peaks_for_fimo.fa'

## Step 1 — Extract sequences → FASTA for FIMO

In [ ]:
genome  = GenomeSequence(FASTA)
regions = list(parse_bed(CUTTAG_BED))
print(f'{len(regions)} peaks')

Path(OUT_FASTA).parent.mkdir(parents=True, exist_ok=True)

# Store (chrom, center) per peak so we can convert FIMO coords back to genome coords
peak_centers = {}   # name → (chrom, center, strand)

with open(OUT_FASTA, 'w') as fa:
    for chrom, start, end, name, strand in regions:
        center = (start + end) // 2
        half   = EXTRACT_LEN // 2
        ws, we = center - half, center + half
        seq    = genome.fetch(chrom, ws, we, EXTRACT_LEN)
        fa.write(f'>{name}\n{seq}\n')
        peak_centers[name] = (chrom, center, strand)

print(f'Written: {OUT_FASTA}')
print()
print('Now run FIMO:')
print(f'  fimo --oc outputs/fimo <motif.meme> {OUT_FASTA}')

## Step 2 — Run FIMO (external)

```bash
fimo --oc outputs/fimo  your_motif.meme  outputs/fimo/peaks_for_fimo.fa
```

Then continue below.

## Step 3 — Parse FIMO output → motif-centered BED

In [ ]:
def parse_fimo_tsv(fimo_tsv: str) -> pd.DataFrame:
    """Parse fimo.tsv, return DataFrame with one best hit per sequence."""
    df = pd.read_csv(fimo_tsv, sep='\t', comment='#')
    # FIMO columns: motif_id, motif_alt_id, sequence_name, start, stop, strand, score, p-value, q-value
    df = df.rename(columns={
        'sequence_name': 'name',
        'start': 'fimo_start',   # 1-based, within extracted sequence
        'stop':  'fimo_stop',
        'p-value': 'pvalue',
        'q-value': 'qvalue',
    })
    df = df.dropna(subset=['name', 'fimo_start', 'fimo_stop'])
    df['fimo_start'] = df['fimo_start'].astype(int)
    df['fimo_stop']  = df['fimo_stop'].astype(int)
    # keep best hit per peak (lowest p-value)
    df = df.sort_values('pvalue').drop_duplicates(subset='name', keep='first')
    return df.set_index('name')


fimo_df = parse_fimo_tsv(FIMO_TSV)
print(f'FIMO hits: {len(fimo_df)} peaks with motif')
fimo_df.head()

In [ ]:
# Convert FIMO positions (within extracted sequence) → genome coordinates
# FIMO uses 1-based coords; motif center = (fimo_start + fimo_stop) / 2 - 1 (0-based offset)

records = []
skipped = 0

for name, (chrom, peak_center, strand) in peak_centers.items():
    if name not in fimo_df.index:
        skipped += 1
        continue

    row = fimo_df.loc[name]
    half = EXTRACT_LEN // 2
    seq_start = peak_center - half   # genome coord of extracted seq start

    # FIMO 1-based → 0-based offset within extracted seq
    motif_offset = (row['fimo_start'] + row['fimo_stop']) / 2 - 1   # 0-based center
    motif_genome = seq_start + int(motif_offset)                     # genome coord

    motif_strand = row.get('strand', '+')
    # For ISM: emit a 1-bp BED entry at motif center; fetch_one_hot will center the window on it
    records.append({
        'chrom':  chrom,
        'start':  motif_genome,
        'end':    motif_genome + 1,
        'name':   name,
        'score':  row.get('score', 0),
        'strand': motif_strand,
        'pvalue': row['pvalue'],
    })

out_df = pd.DataFrame(records)
print(f'Motif-centered entries: {len(out_df)}  |  skipped (no hit): {skipped}')
out_df.head()

In [ ]:
# Write motif-centered BED (BED6 format)
out_df[['chrom', 'start', 'end', 'name', 'score', 'strand']].to_csv(
    OUT_BED, sep='\t', header=False, index=False
)
print(f'Written: {OUT_BED}')
print()
print('Next step: open ism.ipynb and set:')
print(f'  BED = "{OUT_BED}"')

## Step 4 — Sanity check: motif offset distribution

Plot how far each motif center is from the original peak center.
A tight distribution around 0 means motifs are well-centered in peaks.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

offsets = []
for name, (chrom, peak_center, strand) in peak_centers.items():
    if name not in fimo_df.index:
        continue
    row = fimo_df.loc[name]
    half = EXTRACT_LEN // 2
    seq_start = peak_center - half
    motif_offset = (row['fimo_start'] + row['fimo_stop']) / 2 - 1
    motif_genome = seq_start + int(motif_offset)
    offsets.append(motif_genome - peak_center)

offsets = np.array(offsets)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(offsets, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', linewidth=1, linestyle='--', label='peak center')
ax.set_xlabel('Motif center offset from peak center (bp)')
ax.set_ylabel('Count')
ax.set_title(f'FIMO motif offset distribution  (n={len(offsets)})')
ax.legend(fontsize=8)
plt.tight_layout()
out_fig = Path(FIMO_TSV).parent / 'motif_offset_dist.pdf'
fig.savefig(out_fig, bbox_inches='tight')
plt.close()
print(f'Offset  mean={offsets.mean():.1f} bp  median={np.median(offsets):.1f} bp  std={offsets.std():.1f} bp')
print(f'Saved: {out_fig}')